In [6]:
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.types import Overwrite
from time import sleep

# 1. 定义状态
class OverAllState(TypedDict):
    """
    状态定义，规约的方式是add追加合并
    """
    logs: Annotated[list[str], add]
    # 如果出现并行节点，同时更新状态，往下游节点传递的时候，必须要有reducer
    cur_id: Annotated[str, add]

# 2. 定义节点
def node_1(state: OverAllState):
    for k, v in state.items():
        print(f"1. k: {k}, v: {v}")
    return {
        "logs": ["node_1 运行完毕"],
        "cur_id": "node_1"
    }

def node_2(state: OverAllState):
    for k, v in state.items():
        print(f"2. k: {k}, v: {v}")
    return {
        "logs": ["node_2 运行完毕"],
        "cur_id": "node_2"
    }

def node_3(state: OverAllState):
    sleep(1)
    for k, v in state.items():
        print(f"3. k: {k}, v: {v}")
    return {
        "logs": ["node_3 运行完毕"],
        "cur_id": "node_3"
    }

def node_4(state: OverAllState):
    sleep(2)
    for k, v in state.items():
        print(f"4. k: {k}, v: {v}")
    return {
        "logs": ["node_4 运行完毕"],
        "cur_id": "node_4"
    }

builder = StateGraph(state_schema=OverAllState)

builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)
builder.add_node("node_4", node_4)

builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_1", "node_3")
builder.add_edge("node_2", "node_4")
builder.add_edge("node_3", "node_4")
builder.add_edge("node_4", END)

graph = builder.compile()

result = graph.invoke({"logs":["START"], "cur_id": "start"})
print('='* 30, '-> result <-', '='* 30)
print(result)


1. k: logs, v: ['START']
1. k: cur_id, v: start
2. k: logs, v: ['START', 'node_1 运行完毕']
2. k: cur_id, v: startnode_1
3. k: logs, v: ['START', 'node_1 运行完毕']
3. k: cur_id, v: startnode_1
4. k: logs, v: ['START', 'node_1 运行完毕', 'node_2 运行完毕', 'node_3 运行完毕']
4. k: cur_id, v: startnode_1node_2node_3
============================== -> result <- ==============================
{'logs': ['START', 'node_1 运行完毕', 'node_2 运行完毕', 'node_3 运行完毕', 'node_4 运行完毕'], 'cur_id': 'startnode_1node_2node_3node_4'}
